In [6]:
# pip install pysabr

In [7]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq
import matplotlib.pylab as plt
from scipy.optimize import least_squares
import pysabr
from pysabr import Hagan2002LognormalSABR

## Import Dataset

In [8]:
sheets = pd.ExcelFile('Swap and Swaption Markets_Amended.xlsx').sheet_names

In [9]:
df_swaption = pd.read_excel('Swap and Swaption Markets_Amended.xlsx', sheet_name=sheets[3], header = 2)
df_swaption

,Expiry,Tenor,-200bps,-150bps,-100bps,-50bps,-25bps,ATM,+25bps,+50bps,+100bps,+150bps,+200bps
0,1Y,1Y,91.570,62.030,44.130,31.224,26.182,22.50,20.96,21.40,24.34,27.488,30.297
1,1Y,2Y,83.270,61.240,46.570,35.807,31.712,28.72,27.12,26.84,28.51,31.025,33.523
2,1Y,3Y,73.920,56.870,44.770,35.745,32.317,29.78,28.29,27.80,28.77,30.725,32.833
3,1Y,5Y,55.190,44.640,36.510,30.242,27.851,26.07,24.98,24.56,25.12,26.536,28.165
4,1Y,10Y,41.180,35.040,30.207,26.619,25.351,24.47,23.98,23.82,24.25,25.204,26.355
5,5Y,1Y,67.800,49.090,38.400,31.485,29.060,27.26,26.04,25.32,24.94,25.320,25.980
6,5Y,2Y,57.880,46.410,39.033,33.653,31.531,29.83,28.56,27.65,26.71,26.540,26.760
7,5Y,3Y,53.430,44.440,38.180,33.437,31.536,29.98,28.76,27.82,26.67,26.200,26.150
8,5Y,5Y,41.990,36.524,32.326,29.005,27.677,26.60,25.73,25.02,24.06,23.570,23.400
9,5Y,10Y,34.417,30.948,28.148,25.954,25.136,24.51,23.99,23.56,22.91,22.490,22.250


Notes on Swaption Data on first inspection:
* This is lognormal Implied Vol for SOFR Swaptions
* Convection for bps strike is (Forward + Basis Point)

In [10]:
def convert_tenor_to_numeric(series):
    num = series.str.slice(stop = -1).astype(int)
    freq = series.str.slice(start = -1)

    conditions = [freq.str.upper() == 'M', freq.str.upper() == 'Y']
    choices = [num * 30 / 360, num.astype(float)]

    return pd.Series(np.select(conditions, choices, default=np.nan), name=series.name)

In [11]:
def convert_bps_to_numeric(column_series):

    column_names = column_series.str.slice(stop = -3)
    replaced_list = []

    for col in column_names:
        if col == '':
            replaced_list.append(0.0)
        else:
            replaced_list.append(float(col) / 10000)
    
    return replaced_list  

In [12]:
# Extract the expiry and tenor information from the column names and convert them to numeric values
# Imp vol is calculated in percentages, so we need to divide by 100 to get the decimal form
# bps is defined as 1/100 of a percentage point, so we need to divide by 10000 to get the decimal form

df_timings = pd.concat([convert_tenor_to_numeric(df_swaption['Expiry']), convert_tenor_to_numeric(df_swaption['Tenor'])], axis=1)
df_bps = df_swaption.drop(columns = ['Expiry', 'Tenor']) / 100  # e.g. 22.5 -> 0.225
df_bps.columns = convert_bps_to_numeric(df_bps.columns)

display(pd.concat([df_timings, df_bps], axis=1))

,Expiry,Tenor,-0.02,-0.015,-0.01,-0.005,-0.0025,0.0,0.0025,0.005,0.01,0.015,0.02
0,1.0,1.0,0.91570,0.62030,0.44130,0.31224,0.26182,0.2250,0.2096,0.2140,0.2434,0.27488,0.30297
1,1.0,2.0,0.83270,0.61240,0.46570,0.35807,0.31712,0.2872,0.2712,0.2684,0.2851,0.31025,0.33523
2,1.0,3.0,0.73920,0.56870,0.44770,0.35745,0.32317,0.2978,0.2829,0.2780,0.2877,0.30725,0.32833
3,1.0,5.0,0.55190,0.44640,0.36510,0.30242,0.27851,0.2607,0.2498,0.2456,0.2512,0.26536,0.28165
4,1.0,10.0,0.41180,0.35040,0.30207,0.26619,0.25351,0.2447,0.2398,0.2382,0.2425,0.25204,0.26355
5,5.0,1.0,0.67800,0.49090,0.38400,0.31485,0.29060,0.2726,0.2604,0.2532,0.2494,0.25320,0.25980
6,5.0,2.0,0.57880,0.46410,0.39033,0.33653,0.31531,0.2983,0.2856,0.2765,0.2671,0.26540,0.26760
7,5.0,3.0,0.53430,0.44440,0.38180,0.33437,0.31536,0.2998,0.2876,0.2782,0.2667,0.26200,0.26150
8,5.0,5.0,0.41990,0.36524,0.32326,0.29005,0.27677,0.2660,0.2573,0.2502,0.2406,0.23570,0.23400
9,5.0,10.0,0.34417,0.30948,0.28148,0.25954,0.25136,0.2451,0.2399,0.2356,0.2291,0.22490,0.22250


We have to pull back the discount factors to calculate annuities. We will use the OIS backcalculated swap discount rates for now

$$
A_{n+1,N}(t) = \sum^{N}_{i=n+1} \Delta_{i-1} D_{i}(t)
$$

$$
\therefore A_{n+1,N}(0) = \sum^{N}_{i=n+1} \Delta_{i-1} D_{i}(0) = \sum^{N}_{i=n+1} \Delta_{i-1} D(0,T_i)
$$

In [13]:
ois_discount_factors = pd.read_csv('ois_sofr_discounted_factors.csv', index_col=0)
ois_discount_factors = ois_discount_factors.drop(columns = 'Market Rate')
ois_discount_factors = ois_discount_factors.loc[round(ois_discount_factors['Tenor_Year_Numeric']) == ois_discount_factors['Tenor_Year_Numeric']].reset_index(drop = True)
display(ois_discount_factors.head(), ois_discount_factors.tail(1))

,Tenor_Year_Numeric,Discount Factor
0,1.0,0.966224
1,2.0,0.935300
2,3.0,0.903758
3,4.0,0.871806
4,5.0,0.839750


,Tenor_Year_Numeric,Discount Factor
29,30.0,0.289438


In [14]:
# Calculate Annuities
# Discount Factor Table is annual frequency!

def calculate_annuities(df_timings, discount_factors):

    annuity =\
    (
        pd.Series(
                    np.zeros(len(df_timings)),
                    name = 'Annuities'
                )
    )

    for row in df_timings.itertuples():
        annuity[row.Index] = np.sum(discount_factors[int(row[1]):int(row[1]+row[2])])

    return annuity

In [15]:
df_annuities = calculate_annuities(df_timings, ois_discount_factors['Discount Factor'])
pd.concat([df_timings, df_annuities, df_bps], axis = 1)

,Expiry,Tenor,Annuities,-0.02,-0.015,-0.01,-0.005,-0.0025,0.0,0.0025,0.005,0.01,0.015,0.02
0,1.0,1.0,0.935300,0.91570,0.62030,0.44130,0.31224,0.26182,0.2250,0.2096,0.2140,0.2434,0.27488,0.30297
1,1.0,2.0,1.839058,0.83270,0.61240,0.46570,0.35807,0.31712,0.2872,0.2712,0.2684,0.2851,0.31025,0.33523
2,1.0,3.0,2.710864,0.73920,0.56870,0.44770,0.35745,0.32317,0.2978,0.2829,0.2780,0.2877,0.30725,0.32833
3,1.0,5.0,4.358245,0.55190,0.44640,0.36510,0.30242,0.27851,0.2607,0.2498,0.2456,0.2512,0.26536,0.28165
4,1.0,10.0,7.930389,0.41180,0.35040,0.30207,0.26619,0.25351,0.2447,0.2398,0.2382,0.2425,0.25204,0.26355
5,5.0,1.0,0.807631,0.67800,0.49090,0.38400,0.31485,0.29060,0.2726,0.2604,0.2532,0.2494,0.25320,0.25980
6,5.0,2.0,1.583386,0.57880,0.46410,0.39033,0.33653,0.31531,0.2983,0.2856,0.2765,0.2671,0.26540,0.26760
7,5.0,3.0,2.327821,0.53430,0.44440,0.38180,0.33437,0.31536,0.2998,0.2876,0.2782,0.2667,0.26200,0.26150
8,5.0,5.0,3.725251,0.41990,0.36524,0.32326,0.29005,0.27677,0.2660,0.2573,0.2502,0.2406,0.23570,0.23400
9,5.0,10.0,6.723663,0.34417,0.30948,0.28148,0.25954,0.25136,0.2451,0.2399,0.2356,0.2291,0.22490,0.22250


The Forward Prices at time 0 for swaps can also be calculated
$$
S_{n,N}(t) = \frac{D_n(t) - D_N(t)}{A_{n+1,N}}
$$

$$
\therefore S_{n,N}(0) = \frac{D_n(0) - D_N(0)}{A_{n+1,N}} = \frac{D(0,T_n) - D(0,T_N)}{A_{n+1,N}}
$$

In [16]:
# Calculate Forward Rates
# Discount Factors is on annual basis

def calculate_forward_rates(df_timings, annuities, discount_factors):

    df_timings = df_timings.astype(int)

    forward_rates =\
    (
        pd
        .Series(np.full(len(df_annuities), np.nan),
                name = "Forward")
    )

    for row in df_timings.itertuples():
        forward_rates[row.Index] = (discount_factors.iloc[row[1]] - discount_factors.iloc[row[1]+row[2]]) / annuities[row.Index]

    return forward_rates

In [17]:
df_forward_rates = calculate_forward_rates(df_timings, df_annuities, ois_discount_factors['Discount Factor'])

In [18]:
pd.concat([df_timings, df_annuities, df_forward_rates, df_bps], axis = 1)

,Expiry,Tenor,Annuities,Forward,-0.02,-0.015,-0.01,-0.005,-0.0025,0.0,0.0025,0.005,0.01,0.015,0.02
0,1.0,1.0,0.935300,0.033724,0.91570,0.62030,0.44130,0.31224,0.26182,0.2250,0.2096,0.2140,0.2434,0.27488,0.30297
1,1.0,2.0,1.839058,0.034525,0.83270,0.61240,0.46570,0.35807,0.31712,0.2872,0.2712,0.2684,0.2851,0.31025,0.33523
2,1.0,3.0,2.710864,0.035247,0.73920,0.56870,0.44770,0.35745,0.32317,0.2978,0.2829,0.2780,0.2877,0.30725,0.32833
3,1.0,5.0,4.358245,0.036608,0.55190,0.44640,0.36510,0.30242,0.27851,0.2607,0.2498,0.2456,0.2512,0.26536,0.28165
4,1.0,10.0,7.930389,0.039007,0.41180,0.35040,0.30207,0.26619,0.25351,0.2447,0.2398,0.2382,0.2425,0.25204,0.26355
5,5.0,1.0,0.807631,0.039468,0.67800,0.49090,0.38400,0.31485,0.29060,0.2726,0.2604,0.2532,0.2494,0.25320,0.25980
6,5.0,2.0,1.583386,0.039911,0.57880,0.46410,0.39033,0.33653,0.31531,0.2983,0.2856,0.2765,0.2671,0.26540,0.26760
7,5.0,3.0,2.327821,0.040339,0.53430,0.44440,0.38180,0.33437,0.31536,0.2998,0.2876,0.2782,0.2667,0.26200,0.26150
8,5.0,5.0,3.725251,0.041100,0.41990,0.36524,0.32326,0.29005,0.27677,0.2660,0.2573,0.2502,0.2406,0.23570,0.23400
9,5.0,10.0,6.723663,0.042281,0.34417,0.30948,0.28148,0.25954,0.25136,0.2451,0.2399,0.2356,0.2291,0.22490,0.22250


We can also output the actual prices of the swaptions using the Black Scholes Lognormal model, which is the same as the Black76 model for swaptions.

In [19]:
## Default Functions for Black Scholes Lognormal Model. 
## In the event of a Black76 model, set S = F and r = 0.0.
## If there is dividend, set r = r-q where q is the dividend yield.

def BlackScholesLognormalPayer(S, K, r, sigma, T):
    d1 = (np.log(S/K)+(r+sigma**2/2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)


def BlackScholesLognormalReceiver(S, K, r, sigma, T):
    d1 = (np.log(S/K)+(r+sigma**2/2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return K*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)

In [20]:
def calculate_prices(df_bps, df_timings, df_forward_rates, df_annuities):
    F = df_forward_rates
    annuity = df_annuities
    r = 0.0          # Black76 discounting convention

    strike_shifts = df_bps.columns.astype(float).to_list()
    T_vec = df_timings['Expiry'].astype(float).values

    payer_strike_shifts = [shift for shift in strike_shifts if shift >= 0]
    receiver_strike_shifts = [shift for shift in strike_shifts if shift < 0]

    # print(payer_strike_shifts, receiver_strike_shifts)

    payer_prices = pd.DataFrame(index=df_bps.index, columns=strike_shifts, dtype=float).drop(columns = receiver_strike_shifts)
    receiver_prices = pd.DataFrame(index=df_bps.index, columns=strike_shifts, dtype=float).drop(columns = payer_strike_shifts)

    for shift in payer_strike_shifts:
        K = F + shift
        vols = df_bps[shift].astype(float).values

        # print(vols)

        payer_prices[shift] = annuity * BlackScholesLognormalPayer(F, K, r, vols, T_vec)


    for shift in receiver_strike_shifts:
        K = F + shift
        vols = df_bps[shift].astype(float).values

        receiver_prices[shift] = annuity * BlackScholesLognormalReceiver(F, K, r, vols, T_vec)


    return payer_prices, receiver_prices, pd.concat([df_timings, receiver_prices, payer_prices], axis=1)

In [21]:
# Attach expiry/tenor for readability
payer_swaption_prices, receiver_swaption_prices, df_swaption_prices = calculate_prices(df_bps, df_timings, df_forward_rates, df_annuities)
display(df_swaption_prices)

,Expiry,Tenor,-0.02,-0.015,-0.01,-0.005,-0.0025,0.0,0.0025,0.005,0.01,0.015,0.02
0,1.0,1.0,0.001488,0.001299,0.001391,0.001748,0.002129,0.002825,0.001718,0.001128,0.000637,0.000434,0.000322
1,1.0,2.0,0.002504,0.002696,0.003306,0.004547,0.005624,0.007250,0.005031,0.003568,0.002081,0.001416,0.001053
2,1.0,3.0,0.002850,0.003509,0.004719,0.006936,0.008744,0.011310,0.008063,0.005789,0.003294,0.002142,0.001527
3,1.0,5.0,0.001971,0.003138,0.005178,0.009016,0.012162,0.016546,0.011519,0.007961,0.004033,0.002303,0.001457
4,1.0,10.0,0.001430,0.003186,0.006916,0.014780,0.021308,0.030123,0.021567,0.015273,0.007791,0.004235,0.002482
5,5.0,1.0,0.006146,0.005593,0.005707,0.006342,0.006892,0.007633,0.006564,0.005698,0.004468,0.003681,0.003141
6,5.0,2.0,0.009677,0.010296,0.011676,0.013675,0.014957,0.016510,0.014419,0.012632,0.009895,0.008016,0.006691
7,5.0,3.0,0.012778,0.014461,0.016950,0.020236,0.022267,0.024651,0.021610,0.018949,0.014732,0.011710,0.009549
8,5.0,5.0,0.013594,0.017272,0.021973,0.027953,0.031598,0.035802,0.031244,0.027201,0.020652,0.015856,0.012393
9,5.0,10.0,0.017247,0.024435,0.033681,0.045648,0.052999,0.061387,0.053835,0.047115,0.035946,0.027425,0.021045


There are two ways to calibrate the model:

1. Calibrate the model to the prices derived from the implied vol reporting models - i.e.: Output Black from implied vol to price, then use calibrate pricing model based on the outputted prices
2. Calibrate the model based on the implied vol reporting models directly

# Part 2: Swaption Calibration

### A: Displaced-Diffusion Model 

This model can be built on top of the Black lognormal Model

$$
V_{n,N}(0) = P_{n+1, N}(0) \cdot Black\left(\frac{S_{n,N}(0)}{\beta}, K+\frac{1-\beta}{\beta}S_{n,N}(0), \sigma\beta, T \right)
$$

Given the above, this can be optimised. Objective is to optimise for $\beta$ and $\sigma$

In [127]:
(df_bps.head(), df_annuities.head(), df_forward_rates.head(), df_timings.head(), df_swaption_prices.head());

In [22]:
def calibrate_displaced_diffusion(df_timings, df_payer_prices, df_receiver_prices):

    annuity = 1.0
    forward_rate = 1.0

    for row in df_timings.itertuples():
        
        i = row.Index

        

In [23]:
calibrate_displaced_diffusion(df_timings, payer_swaption_prices, receiver_swaption_prices)

In [ ]:
# Displaced Diffusion (Shifted Lognormal) Model Calibration
# Formula: Price = Black(F/beta, K + (1-beta)/beta * F, sigma*beta, T)

def DisplacedDiffusion_price(F, K, T, beta, sigma):
    """
    Calculate price under displaced diffusion model using Black76 formula.
    
    The displaced diffusion model adjusts the forward and strike:
    - Effective Forward: F_eff = F / beta
    - Effective Strike: K_eff = K + (1-beta)/beta * F
    - Effective Volatility: sigma_eff = sigma * beta
    
    Then applies Black's formula.
    """
    if beta <= 0 or beta > 1.0:
        return np.nan
    
    F_eff = F / beta
    K_eff = K + (1 - beta) / beta * F
    sigma_eff = sigma * beta
    
    # Use Black76 formula (put/call doesn't matter for our purposes, using payer convention)
    r = 0.0  # Black76 convention
    price = BlackScholesLognormalPayer(F_eff, K_eff, r, sigma_eff, T)
    
    return price


def DisplacedDiffusion_implied_vol(F, K, T, beta, sigma):
    """
    Calculate implied Black volatility for displaced diffusion model.
    Inverts the price to find what Black vol would produce the same price.
    """
    try:
        # Get price from DD model
        dd_price = DisplacedDiffusion_price(F, K, T, beta, sigma)
        
        if np.isnan(dd_price) or dd_price <= 0:
            return np.nan
        
        # Invert to get implied vol using brentq
        implied_vol = brentq(
            lambda vol: BlackScholesLognormalPayer(F, K, 0.0, vol, T) - dd_price,
            1e-12, 10.0
        )
        return implied_vol
    except:
        return np.nan


def dd_calibration_error(params, strikes, market_vols, F, T):
    """
    Calculate total squared error between market vols and displaced diffusion model vols.
    params = [beta, sigma]
    """
    beta, sigma = params
    
    # Validate bounds
    if beta <= 0 or beta > 1.0 or sigma <= 0:
        return 1e10
    
    error = 0.0
    for strike, market_vol in zip(strikes, market_vols):
        try:
            dd_vol = DisplacedDiffusion_implied_vol(F, strike, T, beta, sigma)
            if np.isnan(dd_vol):
                return 1e10
            error += (market_vol - dd_vol) ** 2
        except:
            return 1e10
    
    return error


def calibrate_beta_via_brentq(sigma, strikes, market_vols, F, T):
    """Use brentq to find beta that minimizes DD model error"""
    def error_func(beta):
        return dd_calibration_error([beta, sigma], strikes, market_vols, F, T)
    
    try:
        beta_opt = brentq(
            lambda b: error_func(b) - error_func(0.5),
            0.01, 0.99
        )
    except:
        beta_opt = 0.5
    
    return beta_opt


def calibrate_sigma_via_brentq(beta, strikes, market_vols, F, T):
    """Use brentq to find sigma that minimizes DD model error"""
    def error_func(sigma):
        return dd_calibration_error([beta, sigma], strikes, market_vols, F, T)
    
    try:
        sigma_opt = brentq(
            lambda s: error_func(s) - error_func(0.3),
            0.001, 2.0
        )
    except:
        sigma_opt = 0.3
    
    return sigma_opt


def calibrate_dd_parameters(strikes, market_vols, F, T, max_iterations=10):
    """Iterative calibration of beta and sigma using brentq"""
    beta_init, sigma_init = 0.5, 0.3
    
    for iteration in range(max_iterations):
        # Optimize each parameter sequentially
        beta_opt = calibrate_beta_via_brentq(sigma_init, strikes, market_vols, F, T)
        sigma_opt = calibrate_sigma_via_brentq(beta_opt, strikes, market_vols, F, T)
        
        # Check convergence
        conv_err = abs(beta_opt - beta_init) + abs(sigma_opt - sigma_init)
        
        if conv_err < 1e-6:
            break
        
        beta_init, sigma_init = beta_opt, sigma_opt
    
    return beta_opt, sigma_opt


# Extract setup parameters
F = 1.0
strikes_list = df_bps.columns.astype(float).to_list()

# Store results for displaced diffusion
dd_calibration_results = []

print("\n" + "=" * 80)
print("DISPLACED DIFFUSION (SHIFTED LOGNORMAL) CALIBRATION")
print("=" * 80)

# Iterate through each expiry-tenor combination
for expiry_idx in range(len(df_timings)):
    T = df_timings.iloc[expiry_idx, 0]  # Time to expiry
    tenor = df_timings.iloc[expiry_idx, 1]  # Tenor
    
    # Extract market implied vols for this expiry
    market_vols = df_bps.iloc[expiry_idx].values
    
    # Create arrays without NaN values
    valid_idx = ~np.isnan(market_vols)
    strikes = np.array(strikes_list)[valid_idx] + F
    market_vols_clean = market_vols[valid_idx]
    
    if len(strikes) > 0:
        print(f"\nExpiry: {T:.4f}Y, Tenor: {tenor:.4f}Y")
        print(f"  Calibrating to {len(strikes)} strikes...")
        
        try:
            # Calibrate parameters
            beta_calib, sigma_calib = calibrate_dd_parameters(
                strikes, market_vols_clean, F, T
            )
            
            # Calculate MSE
            mse_dd = dd_calibration_error(
                [beta_calib, sigma_calib], strikes, market_vols_clean, F, T
            ) / len(strikes)
            
            print(f"  beta={beta_calib:.6f}, sigma={sigma_calib:.6f}, MSE={mse_dd:.2e}")
            
            # Store result
            dd_calibration_results.append({
                'Expiry (Years)': T,
                'Tenor (Years)': tenor,
                'Num Strikes': len(strikes),
                'Beta': beta_calib,
                'Sigma': sigma_calib,
                'MSE': mse_dd
            })
        except Exception as e:
            print(f"  Calibration failed: {str(e)}")
    else:
        print(f"\nExpiry: {T:.4f}Y, Tenor: {tenor:.4f}Y - No valid data")

# Create results dataframe
df_dd_calibration = pd.DataFrame(dd_calibration_results)

print("\n" + "=" * 80)
print("\nDISPLACED DIFFUSION PARAMETERS - SUMMARY TABLE")
print("=" * 80)
display(df_dd_calibration.round(6))



DISPLACED DIFFUSION (SHIFTED LOGNORMAL) CALIBRATION

Expiry: 1.0000Y, Tenor: 1.0000Y
  Calibrating to 11 strikes...
  beta=0.500000, sigma=0.300000, MSE=4.76e-02

Expiry: 1.0000Y, Tenor: 2.0000Y
  Calibrating to 11 strikes...
  beta=0.500000, sigma=0.300000, MSE=3.74e-02

Expiry: 1.0000Y, Tenor: 3.0000Y
  Calibrating to 11 strikes...
  beta=0.500000, sigma=0.300000, MSE=2.63e-02

Expiry: 1.0000Y, Tenor: 5.0000Y
  Calibrating to 11 strikes...
  beta=0.500000, sigma=0.300000, MSE=8.98e-03

Expiry: 1.0000Y, Tenor: 10.0000Y
  Calibrating to 11 strikes...
  beta=0.500000, sigma=0.300000, MSE=3.22e-03

Expiry: 5.0000Y, Tenor: 1.0000Y
  Calibrating to 11 strikes...
  beta=0.500000, sigma=0.300000, MSE=1.75e-02

Expiry: 5.0000Y, Tenor: 2.0000Y
  Calibrating to 11 strikes...
  beta=0.500000, sigma=0.300000, MSE=1.03e-02

Expiry: 5.0000Y, Tenor: 3.0000Y
  Calibrating to 11 strikes...
  beta=0.500000, sigma=0.300000, MSE=7.64e-03

Expiry: 5.0000Y, Tenor: 5.0000Y
  Calibrating to 11 strikes...
  

,Expiry (Years),Tenor (Years),Num Strikes,Beta,Sigma,MSE
0,1.0,1.0,11,0.5,0.3,0.047637
1,1.0,2.0,11,0.5,0.3,0.037427
2,1.0,3.0,11,0.5,0.3,0.026255
3,1.0,5.0,11,0.5,0.3,0.008977
4,1.0,10.0,11,0.5,0.3,0.003219
5,5.0,1.0,11,0.5,0.3,0.017459
6,5.0,2.0,11,0.5,0.3,0.010280
7,5.0,3.0,11,0.5,0.3,0.007636
8,5.0,5.0,11,0.5,0.3,0.003425
9,5.0,10.0,11,0.5,0.3,0.003397


### B: SABR Model

The implied Black volatility of the SABR model is given below, where $\beta = 0.75$ as a default setting

$$
    \begin{split}
      \sigma_{SABR}(F_0, K, \alpha, \beta, \rho, \nu)
      = \frac{\alpha}{(F_0K)^{(1-\beta)/2}\left\{ 1 + \frac{(1-\beta)^2}{24}\log^2\left(\frac{F_0}{K}\right) + \frac{(1-\beta)^4}{1920}\log^4\left(\frac{F_0}{K}\right) + \cdots\right\} }
      \times \frac{z}{x(z)} \times \left\{ 1 + \left[
           \frac{(1-\beta)^2}{24}
           \frac{\alpha^2}{(F_0K)^{1-\beta}}+\frac{1}{4}\frac{\rho\beta\nu\alpha}{(F_0K)^{(1-\beta)/2}}+\frac{2-3\rho^2}{24}\nu^2\right]
         T + \cdots \right.
    \end{split}
$$

where

$$
    \begin{split}
      z = \frac{\nu}{\alpha} (F_0K)^{(1-\beta)/2}
      \log\left(\frac{F_0}{K}\right),
    \end{split}
$$

and

$$
    % \begin{split}
      x(z) = \log \left[ \frac{\sqrt{1-2\rho z+z^2}+z -\rho}{1-\rho}
      \right].
    % \end{split}
$$

In [25]:
def SABR(F, K, T, alpha, beta, rho, nu):
    X = K
    # if K is at-the-money-forward
    if abs(F - K) < 1e-12:
        numer1 = (((1 - beta)**2)/24)*alpha*alpha/(F**(2 - 2*beta))
        numer2 = 0.25*rho*beta*nu*alpha/(F**(1 - beta))
        numer3 = ((2 - 3*rho*rho)/24)*nu*nu
        VolAtm = alpha*(1 + (numer1 + numer2 + numer3)*T)/(F**(1-beta))
        sabrsigma = VolAtm
    else:
        z = (nu/alpha)*((F*X)**(0.5*(1-beta)))*np.log(F/X)
        zhi = np.log((((1 - 2*rho*z + z*z)**0.5) + z - rho)/(1 - rho))
        numer1 = (((1 - beta)**2)/24)*((alpha*alpha)/((F*X)**(1 - beta)))
        numer2 = 0.25*rho*beta*nu*alpha/((F*X)**((1 - beta)/2))
        numer3 = ((2 - 3*rho*rho)/24)*nu*nu
        numer = alpha*(1 + (numer1 + numer2 + numer3)*T)*z
        denom1 = ((1 - beta)**2/24)*(np.log(F/X))**2
        denom2 = (((1 - beta)**4)/1920)*((np.log(F/X))**4)
        denom = ((F*X)**((1 - beta)/2))*(1 + denom1 + denom2)*zhi
        sabrsigma = numer/denom

    return sabrsigma

The definition above contains the function

$$
\begin{split}
{SABR}(F, K, T, \alpha, \beta, \rho, \nu)
\end{split}
$$

The function returns a volatility $\sigma_{{SABR}}$ for the Black76Lognormal call or put option formula, so that

$$
\begin{split}
{Call price} &= {BlackScholesCall}(S, K, r, \sigma_{{SABR}}, T) \\
{Put price} &= {BlackScholesPut}(S, K, r, \sigma_{{SABR}}, T) \\
\end{split}
$$

How do we determine the parameters $\alpha$, $\rho$ and $\nu$?
- We choose them so that the output of the SABR model matches the implied volatilities observed in the market.
- We refer to this process as "model calibration".

In other words, defining

  $$
    \begin{split}
      \sigma_{{Mkt}}(K_1) - {SABR}(F, K_1, T, \alpha, 0.8, \rho, \nu) &= \epsilon_1 \\
      \sigma_{{Mkt}}(K_2) - {SABR}(F, K_2, T, \alpha, 0.8, \rho, \nu) &= \epsilon_2 \\
      \vdots&\\
      \sigma_{{Mkt}}(K_n) - {SABR}(F, K_n, T, \alpha, 0.8, \rho, \nu) &= \epsilon_n \\
    \end{split}
  $$

We want to minimize the sum of squared error terms as follows:
  
  $$
    \begin{split}
      \min_{\substack{\alpha,\; \rho,\; \nu}} \;\sum_{i=1}^n \epsilon_i^2
    \end{split}
  $$

We use the "least_squares" algorithm in "scipy" package to calibrate the SABR model parameters:


A suggested optimisation routine is as follows
1. Fix the $\beta$ value (in this assignment: 0.75)
2. At the ATM values, optimise for alpha
3. Finally, at the shifts, optimise for $\rho$ and $\nu$

The SABR model could be fitted using the library pysabr

In [121]:
df_bps

,-0.0200,-0.0150,-0.0100,-0.0050,-0.0025,0.0000,0.0025,0.0050,0.0100,0.0150,0.0200
0,0.91570,0.62030,0.44130,0.31224,0.26182,0.2250,0.2096,0.2140,0.2434,0.27488,0.30297
1,0.83270,0.61240,0.46570,0.35807,0.31712,0.2872,0.2712,0.2684,0.2851,0.31025,0.33523
2,0.73920,0.56870,0.44770,0.35745,0.32317,0.2978,0.2829,0.2780,0.2877,0.30725,0.32833
3,0.55190,0.44640,0.36510,0.30242,0.27851,0.2607,0.2498,0.2456,0.2512,0.26536,0.28165
4,0.41180,0.35040,0.30207,0.26619,0.25351,0.2447,0.2398,0.2382,0.2425,0.25204,0.26355
5,0.67800,0.49090,0.38400,0.31485,0.29060,0.2726,0.2604,0.2532,0.2494,0.25320,0.25980
6,0.57880,0.46410,0.39033,0.33653,0.31531,0.2983,0.2856,0.2765,0.2671,0.26540,0.26760
7,0.53430,0.44440,0.38180,0.33437,0.31536,0.2998,0.2876,0.2782,0.2667,0.26200,0.26150
8,0.41990,0.36524,0.32326,0.29005,0.27677,0.2660,0.2573,0.2502,0.2406,0.23570,0.23400
9,0.34417,0.30948,0.28148,0.25954,0.25136,0.2451,0.2399,0.2356,0.2291,0.22490,0.22250


In [ ]:
# Example Market Data (Strikes and corresponding Implied Volatilities)

market_strikes = np.array([0.02, 0.025, 0.03]) 
market_vols = np.array([0.45, 0.40, 0.38]) / 100 # convert to decimal

# Market data inputs
forward = 0.025271 # current forward rate/price
shift = 0.03       # shift value for shifted SABR (handles negative rates/prices)
t = 10             # time to maturity (years)
beta = 0.5         # fixed beta parameter

# Initialize the SABR object with fixed beta and shift
sabr = pysabr.Hagan2002LognormalSABR(f=forward, shift=shift, t=t, beta=beta)

# Calibrate alpha, rho, and volvol (nu) from the market data
# The calibrate method finds the parameters that best fit the provided strikes and vols
alpha, rho, volvol = sabr.fit(market_strikes, market_vols)

print(f"Calibrated parameters: alpha={alpha:.4f}, rho={rho:.4f}, volvol={volvol:.4f}")

Calibrated parameters: alpha=0.0001, rho=0.5590, volvol=0.0007


In [122]:
def calibrate_sabr(strikes_shift, vols, forward, tenor, beta, shift=0.0):

    strikes = np.asarray(strikes_shift+forward)
    vols = np.asarray(vols)
    # print(strikes)
    # print(vols)

    sabr = pysabr.Hagan2002LognormalSABR(f = forward, shift = shift, t = tenor, beta = beta, v_atm_n = vols[5])
    alpha, rho, nu = sabr.fit(strikes, vols)

    return alpha, rho, nu

In [123]:
def output_sabr_values(df_bps, forward_series, tenor_series, beta, shift=0.0):
    
    strikes_shift = np.asarray(df_bps.columns)
    
    output_rows = []

    for i in df_bps.index:
        # strikes = strikes_shift + forward_series[i]
        vols = np.asarray(df_bps.iloc[i])
        
        alpha, rho, nu = calibrate_sabr(strikes_shift, vols, forward_series[i], tenor_series[i], beta, shift)

        output_rows.append({
            'alpha': alpha,
            'rho': rho,
            'nu': nu
        })
    
    return pd.DataFrame(
                        output_rows,
                        index = df_bps.index,
                        columns = ['alpha', 'rho', 'nu']
                        )


In [124]:
output_sabr_values(df_bps, df_forward_rates, df_timings['Tenor'], beta = 0.75)

C:\Users\TanFamily4\AppData\Roaming\Python\Python313\site-packages\pysabr\models\hagan_2002_lognormal_sabr.py:82: RuntimeWarning: invalid value encountered in log
  return np.log(a / b)


,alpha,rho,nu
0,0.001588,0.999898,0.000100
1,0.001214,-0.499492,0.018643
2,0.001278,-0.451605,0.015668
3,0.001137,-0.377645,0.011376
4,0.001237,0.999750,0.000100
5,0.001512,0.999899,0.000100
6,0.001527,0.999899,0.000100
7,0.001497,0.999778,0.000100
8,0.001294,0.999849,0.000100
9,0.001171,0.999898,0.000100
